# Model 1 v2: Road Closure Probability Prediction (Refined)

**Objective**: Predict the probability (0-100%) of road closure for traffic events in Bengaluru

**Dataset**: Feature-engineered Astram event data (~8,200 events) from `outputs/feature_engineering_v1/`

**Approach**:
- Binary classification with probabilistic output
- Calibrated probabilities for reliable risk assessment
- Train models independently (LightGBM, XGBoost)
- Optuna hyperparameter optimization (30 trials each)
- Calibration on held-out calibration split (no val-set double-dipping)
- Weighted soft-voting ensemble with validation-selected weights
- Threshold optimization for operational decision-making
- Forward-chaining leakage-safe Model 2 handoff

**Leakage fixes over v1**:
1. Calibration uses a **separate calibration split** carved from training data (not the validation set)
2. Ensemble weight search uses **validation set** which was NOT used for calibration
3. Forward-chaining handoff reuses the **same preprocessor** as the main model
4. All paths are project-relative (no hardcoded Windows paths)

## 1. Import Libraries

In [ ]:
# Core libraries
import re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    confusion_matrix, classification_report,
    brier_score_loss, accuracy_score,
    precision_score, recall_score
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, ClassifierMixin

# Gradient Boosting Models
import lightgbm as lgb
import xgboost as xgb

# Hyperparameter optimization
import optuna

# Explainability
import shap

# Utilities
from datetime import datetime
import joblib
import os
from pathlib import Path

print("All libraries imported successfully!")
print(f"Python environment ready - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Load Feature-Engineered Dataset

In [ ]:
# ── Project-relative paths ────────────────────────────────────────────────────
RANDOM_STATE = 42

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents]
     if (p / 'outputs' / 'feature_engineering_v1').exists()),
    cwd,
)

ROAD_CLOSURE_DATA_PATH = PROJECT_ROOT / 'outputs' / 'feature_engineering_v1' / 'road_closure_features_v1.csv'
DURATION_BASE_DATA_PATH = PROJECT_ROOT / 'outputs' / 'feature_engineering_v1' / 'duration_base_features_v1.csv'

output_dir = PROJECT_ROOT / 'outputs' / 'model_road_closure_v2'
output_dir.mkdir(parents=True, exist_ok=True)

ROAD_CLOSURE_TARGET = 'target_road_closure'

if not ROAD_CLOSURE_DATA_PATH.exists():
    raise FileNotFoundError(f"Missing: {ROAD_CLOSURE_DATA_PATH}")

road_closure_df = pd.read_csv(str(ROAD_CLOSURE_DATA_PATH))
df = road_closure_df.copy()

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {output_dir}")
print(f"Road-closure data: {df.shape}")

## 3. Leakage Audit

This section verifies that blocked post-event fields have not entered the Model 1 input. Statistical checks use only the chronological training portion and produce warnings rather than automatically removing valid predictive features.

In [ ]:
METADATA_COLS = ['id', '_source_row', 'prediction_datetime', 'start_datetime']

BLOCKED_FEATURES = {
    'requires_road_closure', 'target_road_closure_missing',
    'duration_minutes_raw', 'duration_band', 'duration_source',
    'duration_label_available_at', 'has_duration_label',
    'end_datetime', 'closed_datetime', 'resolved_datetime', 'modified_datetime',
    'status', 'closed_by_id', 'resolved_by_id', 'last_modified_by_id',
    'comment', 'endlatitude', 'endlongitude', 'end_address',
    'route_path', 'route_distance_km', 'has_route_span', 'has_end_location',
    'end_point_missing_or_zero', 'resolved_at_address',
    'resolved_at_latitude', 'resolved_at_longitude',
}

raw_feature_cols = [
    col for col in road_closure_df.columns
    if col not in METADATA_COLS + [ROAD_CLOSURE_TARGET]
]

# Hard leakage checks
blocked_present = sorted(set(raw_feature_cols).intersection(BLOCKED_FEATURES))
qa_columns_present = sorted(col for col in raw_feature_cols if col.startswith('qa_'))

if blocked_present:
    raise ValueError(f'Blocked features found: {blocked_present}')
if qa_columns_present:
    raise ValueError(f'QA-only columns found: {qa_columns_present}')

prediction_times = pd.to_datetime(
    road_closure_df['prediction_datetime'], errors='coerce', utc=True,
)
if not prediction_times.dropna().is_monotonic_increasing:
    raise ValueError('Dataset is not ordered by prediction time.')

print('Hard leakage and schema checks passed.')

# Statistical diagnostics: training portion only
audit_train_end = int(0.70 * len(road_closure_df))
audit_train = road_closure_df.iloc[:audit_train_end]
audit_target = audit_train[ROAD_CLOSURE_TARGET].astype(int)

audit_rows = []
for col in raw_feature_cols:
    if not pd.api.types.is_numeric_dtype(audit_train[col]):
        continue
    values = pd.to_numeric(audit_train[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    valid = values.notna() & audit_target.notna()
    if valid.sum() == 0 or values.loc[valid].nunique() <= 1 or audit_target.loc[valid].nunique() != 2:
        continue
    x_valid = values.loc[valid]
    y_valid = audit_target.loc[valid]
    correlation = x_valid.corr(y_valid)
    auc = roc_auc_score(y_valid, x_valid)
    auc_strength = max(auc, 1 - auc)
    suspicious = auc_strength >= 0.995 or (pd.notna(correlation) and abs(correlation) >= 0.98)
    audit_rows.append({
        'feature': col, 'auc_strength': auc_strength,
        'correlation': correlation,
        'status': 'WARN' if suspicious else 'PASS',
        'decision': 'manual_review_only' if suspicious else 'retain',
    })

leakage_audit_df = pd.DataFrame(audit_rows)
LEAKAGE_FEATURES = []
LEAKAGE_REVIEW_FEATURES = sorted(col for col in raw_feature_cols if col.startswith('past_closure_'))

if not leakage_audit_df.empty:
    leakage_audit_df = leakage_audit_df.sort_values(
        ['status', 'auc_strength'], ascending=[False, False]
    ).reset_index(drop=True)

warnings_df = (
    leakage_audit_df[leakage_audit_df['status'].eq('WARN')]
    if not leakage_audit_df.empty else pd.DataFrame()
)

if not warnings_df.empty:
    display(warnings_df)
else:
    print('No WARN or FAIL findings.')

print('Raw Model 1 features:', len(raw_feature_cols))
print('Statistical warnings:', len(warnings_df))

## 4. Data Preparation & Chronological Split

**Key Decision**: Use chronological split (not random) to avoid data leakage across time.

**Leakage fix**: Split into Train (56%) / Calibration (14%) / Validation (15%) / Test (15%). Calibration split is carved from training data so that probability calibration does NOT touch the validation set used for ensemble weight selection and threshold optimization.

In [ ]:
target_col = ROAD_CLOSURE_TARGET

# Parse timestamps for ordering
df['prediction_datetime'] = pd.to_datetime(df['prediction_datetime'], errors='coerce', utc=True)
df['start_datetime'] = pd.to_datetime(df['start_datetime'], errors='coerce', utc=True)

# Ensure stable chronological ordering
df = df.sort_values(
    ['prediction_datetime', 'start_datetime', '_source_row'],
    kind='mergesort', na_position='last',
).reset_index(drop=True)

# Features: exclude metadata, target, and any leakage features
model_input_cols = [
    col for col in df.columns
    if col not in METADATA_COLS + [target_col]
    and col not in LEAKAGE_FEATURES
]

X_raw = df[model_input_cols].copy().replace([np.inf, -np.inf], np.nan)
y = df[target_col].astype(int).copy()

print('Raw Model 1 features:', len(model_input_cols))
print('Raw feature matrix:', X_raw.shape)
print('Target vector:', y.shape)

In [ ]:
# ── Chronological split: Train 56% / Calibration 14% / Validation 15% / Test 15% ──
# LEAKAGE FIX: calibration is carved from training data, NOT from validation.
n = len(X_raw)
train_end = int(0.56 * n)      # first 56% for model training
cal_end   = int(0.70 * n)      # next 14% for probability calibration
val_end   = int(0.85 * n)      # next 15% for weight selection + threshold
                                # final 15% held-out test

X_train_raw = X_raw.iloc[:train_end].copy()
X_cal_raw   = X_raw.iloc[train_end:cal_end].copy()
X_val_raw   = X_raw.iloc[cal_end:val_end].copy()
X_test_raw  = X_raw.iloc[val_end:].copy()

y_train = y.iloc[:train_end].copy()
y_cal   = y.iloc[train_end:cal_end].copy()
y_val   = y.iloc[cal_end:val_end].copy()
y_test  = y.iloc[val_end:].copy()

print('Chronological Split Summary:')
print(f"{'Split':<14} {'Rows':<10} {'Closure Rate':<15}")
for name, ys in [('Train', y_train), ('Calibration', y_cal),
                  ('Validation', y_val), ('Test', y_test)]:
    print(f"{name:<14} {len(ys):<10,} {ys.mean()*100:>6.2f}%")

In [ ]:
# ── Preprocessing: fit on TRAINING rows only ─────────────────────────────────
numeric_cols = X_train_raw.select_dtypes(include=[np.number, 'bool']).columns.tolist()
categorical_cols = [col for col in model_input_cols if col not in numeric_cols]

numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', OneHotEncoder(
        handle_unknown='infrequent_if_exist', min_frequency=20,
        sparse_output=False, dtype=np.float32,
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_cols),
        ('categorical', categorical_pipeline, categorical_cols),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

X_train_transformed = preprocessor.fit_transform(X_train_raw)

def make_safe_unique_names(names):
    safe_names, seen = [], {}
    for name in names:
        base = re.sub(r'[^A-Za-z0-9_]+', '_', str(name)).strip('_') or 'feature'
        occ = seen.get(base, 0); seen[base] = occ + 1
        safe_names.append(f'{base}__{occ}' if occ else base)
    return safe_names

feature_cols = make_safe_unique_names(preprocessor.get_feature_names_out())

def make_feature_frame(transformed, index):
    return pd.DataFrame(transformed, columns=feature_cols, index=index).astype(np.float32)

def transform_features(frame):
    return make_feature_frame(preprocessor.transform(frame), frame.index)

X_train = make_feature_frame(X_train_transformed, X_train_raw.index)
X_cal   = transform_features(X_cal_raw)
X_val   = transform_features(X_val_raw)
X_test  = transform_features(X_test_raw)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print('Numeric raw features:', len(numeric_cols))
print('Categorical raw features:', len(categorical_cols))
print('Encoded model features:', len(feature_cols))
print('Positive class weight:', round(scale_pos_weight, 2))

## 5. Baseline Model - Logistic Regression

In [ ]:
# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE, solver='lbfgs')
lr_model.fit(X_train_scaled, y_train)

lr_val_proba = lr_model.predict_proba(X_val_scaled)[:, 1]

lr_val_auc = roc_auc_score(y_val, lr_val_proba)
lr_val_prauc = average_precision_score(y_val, lr_val_proba)
lr_val_brier = brier_score_loss(y_val, lr_val_proba)

print("Logistic Regression (Baseline) - Validation Set:")
print(f"  ROC-AUC:     {lr_val_auc:.4f}")
print(f"  PR-AUC:      {lr_val_prauc:.4f}")
print(f"  Brier Score: {lr_val_brier:.4f}")

## 6. LightGBM Model (Primary Model)

In [ ]:
# LightGBM default-param run
lgb_params = {
    'objective': 'binary', 'metric': 'average_precision', 'boosting_type': 'gbdt',
    'num_leaves': 31, 'learning_rate': 0.05, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 5, 'scale_pos_weight': scale_pos_weight,
    'verbose': -1, 'seed': RANDOM_STATE, 'deterministic': True,
    'force_col_wise': True, 'n_jobs': -1
}

# LightGBM trains against calibration set for early stopping (NOT validation)
lgb_train = lgb.Dataset(X_train, y_train, params={'feature_pre_filter': False})
lgb_cal   = lgb.Dataset(X_cal, y_cal, reference=lgb_train, params={'feature_pre_filter': False})

print("Training LightGBM (default params)...")
lgb_model = lgb.train(
    lgb_params, lgb_train, num_boost_round=1000,
    valid_sets=[lgb_train, lgb_cal], valid_names=['train', 'cal'],
    callbacks=[lgb.early_stopping(stopping_rounds=50, first_metric_only=True),
               lgb.log_evaluation(period=100)]
)

lgb_val_proba = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)
lgb_val_auc = roc_auc_score(y_val, lgb_val_proba)
lgb_val_prauc = average_precision_score(y_val, lgb_val_proba)

print(f"\nLightGBM - Validation Set:")
print(f"  ROC-AUC: {lgb_val_auc:.4f}")
print(f"  PR-AUC:  {lgb_val_prauc:.4f}")
print(f"  Best iteration: {lgb_model.best_iteration}")

## 7. XGBoost Model

In [ ]:
# XGBoost default-param run
xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'aucpr',
    'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'scale_pos_weight': scale_pos_weight,
    'random_state': RANDOM_STATE, 'n_jobs': -1
}

dtrain = xgb.DMatrix(X_train, label=y_train)
dcal   = xgb.DMatrix(X_cal, label=y_cal)
dval   = xgb.DMatrix(X_val, label=y_val)
dtest  = xgb.DMatrix(X_test, label=y_test)

print("Training XGBoost (default params)...")
xgb_model = xgb.train(
    xgb_params, dtrain, num_boost_round=1000,
    evals=[(dtrain, 'train'), (dcal, 'cal')],
    early_stopping_rounds=50, verbose_eval=100
)

xgb_val_proba = xgb_model.predict(dval, iteration_range=(0, xgb_model.best_iteration + 1))
xgb_val_auc = roc_auc_score(y_val, xgb_val_proba)
xgb_val_prauc = average_precision_score(y_val, xgb_val_proba)

print(f"\nXGBoost - Validation Set:")
print(f"  ROC-AUC: {xgb_val_auc:.4f}")
print(f"  PR-AUC:  {xgb_val_prauc:.4f}")
print(f"  Best iteration: {xgb_model.best_iteration}")

## 8. Hyperparameter Optimization + Ensemble Training

Both LightGBM and XGBoost are independently tuned via Optuna, trained with best params, individually calibrated **on the held-out calibration split**, then combined using a validation-selected soft-voting weight.

**Leakage fix**: Calibration is fitted on `X_cal/y_cal` (rows 56%-70%), NOT on the validation set. Ensemble weights and threshold are then selected on the untouched `X_val/y_val` (rows 70%-85%).

In [ ]:
N_TRIALS = 30
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── LightGBM Optuna study ────────────────────────────────────────────────────
def objective_lgb(trial):
    params = {
        'objective': 'binary', 'metric': 'average_precision',
        'boosting_type': 'gbdt', 'verbosity': -1,
        'seed': RANDOM_STATE, 'deterministic': True, 'force_col_wise': True,
        'num_leaves': trial.suggest_int('num_leaves', 12, 96),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 15, 140),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.55, 1.0),
        'bagging_freq': 1,
        'feature_fraction': trial.suggest_float('feature_fraction', 0.55, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-6, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 20.0, log=True),
        'scale_pos_weight': trial.suggest_float(
            'scale_pos_weight', max(1.0, 0.4 * scale_pos_weight), 1.4 * scale_pos_weight),
    }
    # Early-stop against calibration set
    model = lgb.train(
        params, lgb_train, num_boost_round=1200,
        valid_sets=[lgb_cal],
        callbacks=[lgb.early_stopping(stopping_rounds=100, first_metric_only=True)]
    )
    # Evaluate on VALIDATION (unseen by early-stopping)
    preds = model.predict(X_val, num_iteration=model.best_iteration)
    return average_precision_score(y_val, preds)

print("Tuning LightGBM...")
study_lgb = optuna.create_study(
    direction='maximize', study_name='lightgbm_tuning',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)
best_params_lgb = study_lgb.best_params
print(f"LightGBM Best PR-AUC: {study_lgb.best_value:.4f}")
print(f"Best params: {best_params_lgb}\n")

# ── XGBoost Optuna study ─────────────────────────────────────────────────────
def objective_xgb(trial):
    params = {
        'objective': 'binary:logistic', 'eval_metric': 'aucpr',
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
        'subsample': trial.suggest_float('subsample', 0.55, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.55, 1.0),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 20.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-6, 3.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-6, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 20.0, log=True),
        'scale_pos_weight': trial.suggest_float(
            'scale_pos_weight', max(1.0, 0.4 * scale_pos_weight), 1.4 * scale_pos_weight),
        'random_state': RANDOM_STATE, 'n_jobs': -1
    }
    # Early-stop against calibration set
    model = xgb.train(
        params, dtrain, num_boost_round=700,
        evals=[(dcal, 'cal')],
        early_stopping_rounds=50, verbose_eval=False
    )
    # Evaluate on VALIDATION
    preds = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
    return average_precision_score(y_val, preds)

print("Tuning XGBoost...")
study_xgb = optuna.create_study(
    direction='maximize', study_name='xgboost_tuning',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE + 1),
)
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)
best_params_xgb = study_xgb.best_params
print(f"XGBoost Best PR-AUC: {study_xgb.best_value:.4f}")
print(f"Best params: {best_params_xgb}")

In [ ]:
# ── Wrapper & calibration classes ────────────────────────────────────────────
class XGBWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model):
        self.model = model
        self.classes_ = np.array([0, 1])
        self._estimator_type = 'classifier'
    def fit(self, X, y=None): return self
    def predict_proba(self, X):
        dmat = xgb.DMatrix(X)
        proba = self.model.predict(dmat, iteration_range=(0, self.model.best_iteration + 1))
        return np.vstack([1 - proba, proba]).T
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

class LGBMWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model, feature_names):
        self.model = model
        self.feature_names = feature_names
        self.classes_ = np.array([0, 1])
        self.n_features_in_ = len(feature_names)
        self._estimator_type = 'classifier'
    def fit(self, X, y=None): return self
    def predict_proba(self, X):
        X_df = pd.DataFrame(X, columns=self.feature_names) if isinstance(X, np.ndarray) else X.copy()
        proba = self.model.predict(X_df, num_iteration=self.model.best_iteration)
        return np.vstack([1 - proba, proba]).T
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

class SigmoidCalibratedModel(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator):
        self.base_estimator = base_estimator
        self.calibrator = LogisticRegression(solver='lbfgs')
        self.classes_ = np.array([0, 1])
        self._estimator_type = 'classifier'
    def fit(self, X, y):
        raw_probs = self.base_estimator.predict_proba(X)[:, 1].reshape(-1, 1)
        self.calibrator.fit(raw_probs, y)
        return self
    def predict_proba(self, X):
        raw_probs = self.base_estimator.predict_proba(X)[:, 1].reshape(-1, 1)
        calibrated_probs = self.calibrator.predict_proba(raw_probs)[:, 1]
        return np.vstack([1 - calibrated_probs, calibrated_probs]).T
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

class EnsembleModel(BaseEstimator, ClassifierMixin):
    """Soft-voting ensemble with validation-selected model weights."""
    def __init__(self, model_lgb, model_xgb, lgb_weight=0.5, xgb_weight=0.5):
        self.model_lgb = model_lgb
        self.model_xgb = model_xgb
        self.lgb_weight = lgb_weight
        self.xgb_weight = xgb_weight
        self.classes_ = np.array([0, 1])
        self._estimator_type = 'classifier'
    def fit(self, X, y=None): return self
    def predict_proba(self, X):
        p_lgb = self.model_lgb.predict_proba(X)[:, 1]
        p_xgb = self.model_xgb.predict_proba(X)[:, 1]
        ensemble = self.lgb_weight * p_lgb + self.xgb_weight * p_xgb
        return np.vstack([1 - ensemble, ensemble]).T
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

# ── Train final LightGBM ─────────────────────────────────────────────────────
print("Training final LightGBM with best params...")
lgb_final_params = {
    'objective': 'binary', 'metric': 'average_precision',
    'boosting_type': 'gbdt', 'bagging_freq': 1,
    'scale_pos_weight': scale_pos_weight, 'verbose': -1,
    'seed': RANDOM_STATE, 'deterministic': True, 'force_col_wise': True,
    **best_params_lgb
}
final_lgb_model = lgb.train(
    lgb_final_params, lgb_train, num_boost_round=1200,
    valid_sets=[lgb_train, lgb_cal], valid_names=['train', 'cal'],
    callbacks=[lgb.early_stopping(stopping_rounds=100, first_metric_only=True),
               lgb.log_evaluation(period=100)]
)
lgb_prefit = LGBMWrapper(final_lgb_model, feature_cols)

# LEAKAGE FIX: calibrate on CALIBRATION set, NOT validation
calibrated_lgb = SigmoidCalibratedModel(lgb_prefit)
calibrated_lgb.fit(X_cal, y_cal)

lgb_cal_val  = calibrated_lgb.predict_proba(X_val)[:, 1]
lgb_raw_val  = lgb_prefit.predict_proba(X_val)[:, 1]

lgb_prauc_raw = average_precision_score(y_val, lgb_raw_val)
lgb_prauc_cal = average_precision_score(y_val, lgb_cal_val)
lgb_brier_raw = brier_score_loss(y_val, lgb_raw_val)
lgb_brier_cal = brier_score_loss(y_val, lgb_cal_val)
print(f"LightGBM  PR-AUC raw={lgb_prauc_raw:.4f}  cal={lgb_prauc_cal:.4f}")
print(f"LightGBM  Brier  raw={lgb_brier_raw:.4f}  cal={lgb_brier_cal:.4f}")

# ── Train final XGBoost ──────────────────────────────────────────────────────
print("\nTraining final XGBoost with best params...")
xgb_final_params = {
    'objective': 'binary:logistic', 'eval_metric': 'aucpr',
    'scale_pos_weight': scale_pos_weight, 'random_state': RANDOM_STATE, 'n_jobs': -1,
    **best_params_xgb
}
final_xgb_model = xgb.train(
    xgb_final_params, dtrain, num_boost_round=1000,
    evals=[(dtrain, 'train'), (dcal, 'cal')],
    early_stopping_rounds=50, verbose_eval=100
)
xgb_prefit = XGBWrapper(final_xgb_model)

# LEAKAGE FIX: calibrate on CALIBRATION set
calibrated_xgb = SigmoidCalibratedModel(xgb_prefit)
calibrated_xgb.fit(X_cal, y_cal)

xgb_cal_val  = calibrated_xgb.predict_proba(X_val)[:, 1]
xgb_raw_val  = xgb_prefit.predict_proba(X_val)[:, 1]

xgb_prauc_raw = average_precision_score(y_val, xgb_raw_val)
xgb_prauc_cal = average_precision_score(y_val, xgb_cal_val)
xgb_brier_raw = brier_score_loss(y_val, xgb_raw_val)
xgb_brier_cal = brier_score_loss(y_val, xgb_cal_val)
print(f"XGBoost   PR-AUC raw={xgb_prauc_raw:.4f}  cal={xgb_prauc_cal:.4f}")
print(f"XGBoost   Brier  raw={xgb_brier_raw:.4f}  cal={xgb_brier_cal:.4f}")

# ── Build ensemble (weight search on VALIDATION — untouched by calibration) ──
ensemble_weight_rows = []
for lgb_weight in np.linspace(0.0, 1.0, 21):
    xgb_weight = 1.0 - lgb_weight
    weighted = lgb_weight * lgb_cal_val + xgb_weight * xgb_cal_val
    ensemble_weight_rows.append({
        'lgb_weight': float(lgb_weight),
        'xgb_weight': float(xgb_weight),
        'validation_pr_auc': float(average_precision_score(y_val, weighted)),
    })

ensemble_weight_results = pd.DataFrame(ensemble_weight_rows)
best_weight_row = ensemble_weight_results.loc[ensemble_weight_results['validation_pr_auc'].idxmax()]
LGB_WEIGHT = float(best_weight_row['lgb_weight'])
XGB_WEIGHT = float(best_weight_row['xgb_weight'])

calibrated_model = EnsembleModel(calibrated_lgb, calibrated_xgb, lgb_weight=LGB_WEIGHT, xgb_weight=XGB_WEIGHT)

ensemble_val_proba  = calibrated_model.predict_proba(X_val)[:, 1]
ensemble_test_proba = calibrated_model.predict_proba(X_test)[:, 1]

# Aliases
calibrated_val_proba  = ensemble_val_proba
calibrated_test_proba = ensemble_test_proba

raw_val_proba = LGB_WEIGHT * lgb_raw_val + XGB_WEIGHT * xgb_raw_val
before_brier = brier_score_loss(y_val, raw_val_proba)
after_brier  = brier_score_loss(y_val, ensemble_val_proba)
before_prauc = average_precision_score(y_val, raw_val_proba)
after_prauc  = average_precision_score(y_val, ensemble_val_proba)

# For SHAP (LightGBM used as interpreter)
final_model_raw = final_lgb_model

print(f"\nSelected ensemble weights: LGB={LGB_WEIGHT:.2f}, XGB={XGB_WEIGHT:.2f}")
print(f"Ensemble  PR-AUC (val):  {after_prauc:.4f}")
print(f"Ensemble  Brier  (val):  {after_brier:.4f}")

## 9. Calibration Review

Calibration is fitted on the **calibration split** (Section 8); this section visualizes the impact on the **validation set** (which was NOT used for fitting the calibrator).

In [ ]:
print("Calibration completed in Section 8 (fitted on calibration split, evaluated on validation).\n")
print(f"{'Model':<18} {'PR-AUC (raw)':<16} {'PR-AUC (cal)':<16} {'Brier (raw)':<14} {'Brier (cal)'}")
print("="*76)
print(f"{'LightGBM':<18} {lgb_prauc_raw:<16.4f} {lgb_prauc_cal:<16.4f} {lgb_brier_raw:<14.4f} {lgb_brier_cal:.4f}")
print(f"{'XGBoost':<18} {xgb_prauc_raw:<16.4f} {xgb_prauc_cal:<16.4f} {xgb_brier_raw:<14.4f} {xgb_brier_cal:.4f}")
print(f"{'Ensemble':<18} {before_prauc:<16.4f} {after_prauc:<16.4f} {before_brier:<14.4f} {after_brier:.4f}")

In [ ]:
# Calibration curve visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for proba, label, color in [
    (lgb_cal_val,        'LightGBM (calibrated)', '#3498db'),
    (xgb_cal_val,        'XGBoost (calibrated)',  '#e67e22'),
    (ensemble_val_proba, 'Ensemble',              '#2ecc71'),
]:
    frac_pos, mean_pred = calibration_curve(y_val, proba, n_bins=10, strategy='quantile')
    axes[0].plot(mean_pred, frac_pos, marker='o', linewidth=2, label=label, color=color)

axes[0].plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated', alpha=0.5)
axes[0].set_xlabel('Mean Predicted Probability', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Fraction of Positives', fontsize=12, fontweight='bold')
axes[0].set_title('Calibration Curve (Validation Set)', fontsize=14, fontweight='bold')
axes[0].legend(loc='best')
axes[0].grid(alpha=0.3)

axes[1].hist(lgb_cal_val,        bins=50, alpha=0.5, label='LightGBM', color='#3498db', density=True)
axes[1].hist(xgb_cal_val,        bins=50, alpha=0.5, label='XGBoost',  color='#e67e22', density=True)
axes[1].hist(ensemble_val_proba, bins=50, alpha=0.5, label='Ensemble', color='#2ecc71', density=True)
axes[1].set_xlabel('Predicted Probability', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Density', fontsize=12, fontweight='bold')
axes[1].set_title('Probability Distribution (Validation)', fontsize=14, fontweight='bold')
axes[1].legend(loc='best')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("Calibration analysis complete")

## 10. Threshold Optimization

Select the operating threshold on validation data by maximizing F1, which balances closure precision and recall.

In [ ]:
# Calculate Precision-Recall curve on validation set
precision, recall, thresholds = precision_recall_curve(y_val, calibrated_val_proba)

# Calculate F1 scores for all thresholds
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)
max_f1 = float(f1_scores.max())
F1_PLATEAU_TOLERANCE = 0.01
plateau_indices = np.flatnonzero(f1_scores >= max_f1 - F1_PLATEAU_TOLERANCE)
best_f1_idx = int(plateau_indices[-1])

# Choose the highest threshold on the near-optimal F1 plateau
optimal_threshold = thresholds[best_f1_idx]

print("Threshold Analysis (Balanced Approach):")
print("="*70)
print(f" Selected Optimal Threshold: {optimal_threshold:.4f}")
print(f"   (Highest threshold within {F1_PLATEAU_TOLERANCE:.2f} of maximum validation F1)")

val_preds_at_thresh = (calibrated_val_proba >= optimal_threshold).astype(int)
print(f"\nValidation Metrics at {optimal_threshold:.4f}:")
print(f"   Precision: {precision_score(y_val, val_preds_at_thresh):.4f}")
print(f"   Recall:    {recall_score(y_val, val_preds_at_thresh):.4f}")
print(f"   F1:        {max_f1:.4f}")
print("="*70)

In [ ]:
# Visualize Precision-Recall tradeoff
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(recall, precision, linewidth=2, color='#3498db', label=f'PR Curve (AUC={after_prauc:.4f})')
axes[0].scatter([recall[best_f1_idx]], [precision[best_f1_idx]], s=200, c='green',
                marker='D', zorder=5, label=f'Optimal F1 (Thresh={optimal_threshold:.2f})')
axes[0].set_xlabel('Recall', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Precision', fontsize=12, fontweight='bold')
axes[0].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[0].legend(loc='best')
axes[0].grid(alpha=0.3)

axes[1].plot(thresholds, precision[:-1], label='Precision', linewidth=2, color='#2ecc71')
axes[1].plot(thresholds, recall[:-1], label='Recall', linewidth=2, color='#e74c3c')
axes[1].plot(thresholds, f1_scores, label='F1-Score', linewidth=2, color='#9b59b6')
axes[1].axvline(optimal_threshold, color='green', linestyle='--', alpha=0.7,
                label=f'Threshold ({optimal_threshold:.4f})')
axes[1].set_xlabel('Threshold', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[1].set_title('Metrics vs Threshold', fontsize=14, fontweight='bold')
axes[1].legend(loc='best')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Test Set Evaluation

In [ ]:
# Final evaluation on TEST set using the ensemble
test_probs    = ensemble_test_proba
test_roc_auc  = roc_auc_score(y_test, test_probs)
test_prauc    = average_precision_score(y_test, test_probs)
test_brier    = brier_score_loss(y_test, test_probs)
test_preds    = (test_probs >= optimal_threshold).astype(int)
test_acc      = accuracy_score(y_test, test_preds)
cm            = confusion_matrix(y_test, test_preds)

print("="*70)
print("FINAL ENSEMBLE (LightGBM + XGBoost) EVALUATION - TEST SET")
print("="*70)
print(f"ROC-AUC:     {test_roc_auc:.4f}")
print(f"PR-AUC:      {test_prauc:.4f}")
print(f"Brier Score: {test_brier:.4f}")
print(f"Accuracy:    {test_acc:.4f} (at {optimal_threshold:.4f} threshold)")
print(f"\nUsing calculated optimal F1-threshold: {optimal_threshold:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=['No Closure', 'Road Closure']))
print("\nConfusion Matrix:")
print(cm)
print("="*70)

# Aliases used by downstream cells
test_probs_xgb   = test_probs
test_roc_auc_xgb = test_roc_auc
test_prauc_xgb   = test_prauc

In [ ]:
# Confusion matrix visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Confusion Matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Closure', 'Road Closure'],
            yticklabels=['No Closure', 'Road Closure'])
axes[0].set_xlabel('Predicted', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual', fontsize=12, fontweight='bold')
axes[0].set_title('Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')

# Plot 2: ROC Curve
fpr, tpr, _ = roc_curve(y_test, test_probs)
axes[1].plot(fpr, tpr, linewidth=2, color='#3498db', label=f'ROC Curve (AUC={test_roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
axes[1].set_title('ROC Curve (Test Set)', fontsize=14, fontweight='bold')
axes[1].legend(loc='best')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Feature Importance Analysis

In [ ]:
# Feature importance for both ensemble components
X_train_lgb = X_train  # LightGBM training frame

lgb_importance = pd.DataFrame({
    'feature':    X_train_lgb.columns,
    'importance': final_lgb_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False).reset_index(drop=True)

xgb_importance_dict = final_xgb_model.get_score(importance_type='gain')
xgb_importance = pd.DataFrame({
    'feature':    list(xgb_importance_dict.keys()),
    'importance': list(xgb_importance_dict.values())
}).sort_values('importance', ascending=False).reset_index(drop=True)

top_lgb = lgb_importance.head(20).copy()
top_xgb = xgb_importance.head(20).copy()

print("Top 20 Features - LightGBM:")
print(top_lgb.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print("\nTop 20 Features - XGBoost:")
print(top_xgb.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].barh(top_lgb['feature'][::-1], top_lgb['importance'][::-1], color='#3498db', alpha=0.8)
axes[0].set_xlabel('Gain', fontsize=12, fontweight='bold')
axes[0].set_title('Top 20 Feature Importance - LightGBM', fontsize=13, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(top_xgb['feature'][::-1], top_xgb['importance'][::-1], color='#e67e22', alpha=0.8)
axes[1].set_xlabel('Gain', fontsize=12, fontweight='bold')
axes[1].set_title('Top 20 Feature Importance - XGBoost', fontsize=13, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Use LightGBM importance as the shared reference (also used by SHAP)
feature_importance = lgb_importance
top_features = top_lgb
print("\nFeature importance for both ensemble components shown")

## 13. SHAP Explainability

Understand **why** specific probabilities are predicted

In [ ]:
# Create SHAP explainer (subsample for speed)
print("Creating SHAP explainer...")
sample_size = min(500, len(X_test))
X_test_sample = X_test.sample(n=sample_size, random_state=42)

explainer = shap.TreeExplainer(final_model_raw)
shap_values = explainer.shap_values(X_test_sample)

print(f"SHAP values computed for {sample_size} test samples")

In [ ]:
# SHAP Summary Plot - LightGBM component (tree-level interpretability)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, feature_names=feature_cols,
                  max_display=20, show=False)
plt.title('SHAP Feature Importance - LightGBM (Ensemble Component)',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()
print("SHAP summary plot generated")

In [ ]:
# Example: Explain specific high-risk and low-risk predictions
test_probas_sample = calibrated_model.predict_proba(X_test_sample)[:, 1]

high_risk_idx = test_probas_sample.argmax()
high_risk_prob = test_probas_sample[high_risk_idx]

low_risk_idx = test_probas_sample.argmin()
low_risk_prob = test_probas_sample[low_risk_idx]

print("Explaining individual predictions:\n")
print(f"High-Risk Event: {high_risk_prob*100:.2f}% closure probability")
print(f"Low-Risk Event:  {low_risk_prob*100:.2f}% closure probability")

# Waterfall plot for high-risk event
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[high_risk_idx],
        base_values=explainer.expected_value,
        data=X_test_sample.iloc[high_risk_idx],
        feature_names=feature_cols
    ),
    max_display=15,
    show=True
)

print("\nIndividual prediction explanation generated")

## 14. Save Final Model & Artifacts

In [ ]:
# Save one self-contained Model 1 inference bundle
bundle_path = str(output_dir / 'model1_inference_bundle.pkl')
model1_output_path = str(output_dir / 'model1_road_closure_predictions.csv')

# Build the single Model 1 output CSV
test_results = pd.DataFrame({
    'actual_label':           y_test.values,
    'predicted_label':        (ensemble_test_proba >= optimal_threshold).astype(int),
    'lgb_probability':        calibrated_lgb.predict_proba(X_test)[:, 1],
    'xgb_probability':        calibrated_xgb.predict_proba(X_test)[:, 1],
    'ensemble_probability':   ensemble_test_proba,
    'prediction_probability': ensemble_test_proba,
    'prediction_percentage':  ensemble_test_proba * 100
}, index=y_test.index)

# Assemble and save the Model 1 test predictions
test_metadata = df.iloc[val_end:][METADATA_COLS].reset_index(drop=True)
csv_export = pd.concat([
    test_metadata,
    test_results.reset_index(drop=True),
], axis=1)
csv_export.to_csv(model1_output_path, index=False)

# Generate feature importance using the selected ensemble weights
ens_imp = lgb_importance.copy()
ens_imp.rename(columns={'importance': 'lgb_importance'}, inplace=True)

xgb_imp_rename = xgb_importance.copy()
xgb_imp_rename.rename(columns={'importance': 'xgb_importance'}, inplace=True)
ens_imp = ens_imp.merge(xgb_imp_rename, on='feature', how='outer')

ens_imp['lgb_importance'] = ens_imp['lgb_importance'].fillna(0)
ens_imp['xgb_importance'] = ens_imp['xgb_importance'].fillna(0)
lgb_importance_total = ens_imp['lgb_importance'].sum()
xgb_importance_total = ens_imp['xgb_importance'].sum()
ens_imp['lgb_importance_normalized'] = ens_imp['lgb_importance'] / max(lgb_importance_total, 1e-12)
ens_imp['xgb_importance_normalized'] = ens_imp['xgb_importance'] / max(xgb_importance_total, 1e-12)
ens_imp['ensemble_importance'] = (
    LGB_WEIGHT * ens_imp['lgb_importance_normalized']
    + XGB_WEIGHT * ens_imp['xgb_importance_normalized']
)
ens_imp = ens_imp.sort_values('ensemble_importance', ascending=False).reset_index(drop=True)
ensemble_importance = ens_imp

# Threshold metadata
threshold_metadata_df = pd.DataFrame([{
    'selection_metric': 'validation_f1_high_threshold_plateau',
    'optimal_threshold': float(optimal_threshold),
    'f1_plateau_tolerance': float(F1_PLATEAU_TOLERANCE),
    'lgb_weight': float(LGB_WEIGHT),
    'xgb_weight': float(XGB_WEIGHT),
    'validation_pr_auc': float(after_prauc),
    'test_roc_auc': float(test_roc_auc),
    'test_pr_auc': float(test_prauc),
    'test_brier': float(test_brier),
    'test_accuracy': float(test_acc),
}])

# Store only library-native fitted objects so the bundle loads in a clean process
portable_model = {
    'type': 'calibrated_lgb_xgb_ensemble',
    'lgb_booster': final_lgb_model,
    'xgb_booster': final_xgb_model,
    'lgb_calibrator': calibrated_lgb.calibrator,
    'xgb_calibrator': calibrated_xgb.calibrator,
    'lgb_best_iteration': int(final_lgb_model.best_iteration),
    'xgb_best_iteration': int(final_xgb_model.best_iteration),
    'lgb_weight': float(LGB_WEIGHT),
    'xgb_weight': float(XGB_WEIGHT),
}

model_bundle = {
    'model': portable_model,
    'preprocessor': preprocessor,
    'model_input_cols': model_input_cols,
    'encoded_feature_cols': feature_cols,
    'optimal_threshold': float(optimal_threshold),
    'lgb_weight': float(LGB_WEIGHT),
    'xgb_weight': float(XGB_WEIGHT),
    'validation_pr_auc': float(after_prauc),
    'test_roc_auc': float(test_roc_auc),
    'test_pr_auc': float(test_prauc),
    'test_brier': float(test_brier),
    'test_accuracy': float(test_acc),
    'leakage_fix': 'calibration_on_separate_split_not_validation',
}
joblib.dump(model_bundle, bundle_path)

# Keep alias for downstream cells
calibrated_probs_test = ensemble_test_proba

print(f"model1_inference_bundle.pkl saved to '{bundle_path}'")
print(f"model1_road_closure_predictions.csv saved to '{model1_output_path}'")
print(f"Decision threshold: {optimal_threshold:.4f}")

In [ ]:
# Verify all output files are in place
files_to_check = [
    ('model1_inference_bundle.pkl',          bundle_path),
    ('model1_road_closure_predictions.csv',  model1_output_path),
]

print("\n" + "="*70)
print("OUTPUT FILES VERIFICATION")
print("="*70)
for file_name, file_path in files_to_check:
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path)
        print(f"  {file_name:<40} {file_size:>10,} bytes")
    else:
        print(f"  {file_name:<40} NOT FOUND")

print(f"\nAll outputs saved to: {output_dir}")
print("="*70)

# Show threshold metadata and data preview
print("\n--- Threshold Metadata ---")
display(threshold_metadata_df)

print("\n--- Final CSV Data Preview (First 5 Rows) ---")
display(csv_export.head())

print(f"\n--- Verification of Ensemble Prediction Granularity ---")
unique_vals = csv_export['ensemble_probability'].nunique()
total_rows = len(csv_export)
print(f"Unique ensemble percentage values: {unique_vals} / {total_rows}")

### Final Project Validation Summary

**1. Dataset & Splitting**
- **Split Method**: Chronological (Time-based).
- **LEAKAGE FIX**: Calibration split (14%) carved from training data, not validation.

**2. Ensemble Model & Performance**
- **Models**: LightGBM + XGBoost, both independently tuned (30 Optuna trials each).
- **Combination**: Validation-selected weighted soft voting; a zero weight is allowed when one component is clearly better.
- **Calibration**: Sigmoid (Platt Scaling) applied independently to each model **on calibration split**.
- **Ensemble weights & threshold**: Selected on **validation set** (untouched by calibration).
- **Test Metrics**: Produced in Section 11 for the ensemble.

**3. Artifacts**
- `outputs/model_road_closure_v2/model1_inference_bundle.pkl`
- `outputs/model_road_closure_v2/model1_road_closure_predictions.csv`
- `outputs/model_road_closure_v2/model2_duration_handoff.csv`

## 15. Production-Ready Prediction Function

In [ ]:
def predict_road_closure_probability(event_features, bundle=None):
    """
    Predict road closure probability using the LightGBM + XGBoost ensemble.
    Accepts raw Model 1 feature-engineering output and applies the fitted
    preprocessor before returning component and weighted probabilities.
    Pass a bundle loaded with joblib.load(...) for use outside this notebook.
    """
    active_bundle = model_bundle if bundle is None else bundle
    raw_columns = active_bundle['model_input_cols']
    missing_cols = sorted(set(raw_columns) - set(event_features.columns))
    if missing_cols:
        raise ValueError(f'Missing Model 1 input columns: {missing_cols}')

    if len(event_features) != 1:
        raise ValueError('Pass exactly one event row to this helper.')

    transformed = active_bundle['preprocessor'].transform(
        event_features[raw_columns].copy()
    )
    encoded_features = pd.DataFrame(
        transformed,
        columns=active_bundle['encoded_feature_cols'],
        index=event_features.index,
    ).astype(np.float32)

    saved_model = active_bundle['model']
    lgb_raw = saved_model['lgb_booster'].predict(
        encoded_features,
        num_iteration=saved_model['lgb_best_iteration'],
    )
    xgb_raw = saved_model['xgb_booster'].predict(
        xgb.DMatrix(encoded_features),
        iteration_range=(0, saved_model['xgb_best_iteration'] + 1),
    )
    p_lgb = saved_model['lgb_calibrator'].predict_proba(
        np.asarray(lgb_raw).reshape(-1, 1)
    )[:, 1][0]
    p_xgb = saved_model['xgb_calibrator'].predict_proba(
        np.asarray(xgb_raw).reshape(-1, 1)
    )[:, 1][0]
    p_ensemble = (
        saved_model['lgb_weight'] * p_lgb
        + saved_model['xgb_weight'] * p_xgb
    )
    percentage = p_ensemble * 100

    if percentage >= 75:
        risk_level = "CRITICAL"
    elif percentage >= 50:
        risk_level = "HIGH"
    elif percentage >= 25:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    threshold = active_bundle['optimal_threshold']
    action = "DEPLOY_RESOURCES" if p_ensemble >= threshold else "MONITOR"

    return {
        'probability_percent': round(percentage, 2),
        'risk_level':          risk_level,
        'recommended_action':  action,
        'ensemble_probability': round(float(p_ensemble), 4),
        'lgb_probability':     round(float(p_lgb), 4),
        'xgb_probability':     round(float(p_xgb), 4),
        'decision_threshold':  round(float(threshold), 4),
        'lgb_weight':           round(float(saved_model['lgb_weight']), 2),
        'xgb_weight':           round(float(saved_model['xgb_weight']), 2),
        'model_used':          'lgb_xgb_ensemble'
    }

print("Ensemble prediction function ready")

In [ ]:
# Example: Test prediction function with a random test sample
sample_event = X_test_raw.sample(1, random_state=RANDOM_STATE)
actual_outcome = y_test[sample_event.index[0]]

prediction = predict_road_closure_probability(sample_event)
print(f"Sample event actual outcome: {'Road Closure' if actual_outcome else 'No Closure'}")
for k, v in prediction.items():
    print(f"  {k}: {v}")

## 16. Leakage-Safe Model 2 Handoff

Creates the duration-model input by joining Model 2's duration-history features with forward-only Model 1 probabilities. Each scored block is predicted by a closure model trained and calibrated only on earlier events; the initial warm-up rows use the already-safe past global closure rate.

**Leakage fix**: The forward-chaining loop now reuses the **same `preprocessor`** fitted in Section 4, ensuring consistent feature encoding. Each fold trains a LightGBM on strictly earlier data, calibrates on a held-out calibration portion within that fold, then scores the next chronological block.

In [ ]:
duration_base_df = pd.read_csv(str(DURATION_BASE_DATA_PATH))

# Forward-chaining probabilities prevent Model 2 from receiving an in-sample
# prediction from a Model 1 that was trained on the same event.
forward_probability = pd.Series(np.nan, index=df.index, dtype=float)
probability_is_history_fallback = pd.Series(0, index=df.index, dtype=int)

warmup_end = int(0.20 * len(df))
fold_edges = np.linspace(warmup_end, len(df), 6, dtype=int)

# The warm-up has insufficient prior rows for a stable fitted model. Its
# strict past-only global rate is a safe fallback and never uses the row's target.
forward_probability.iloc[:warmup_end] = (
    pd.to_numeric(df['past_closure_global_rate'], errors='coerce')
    .iloc[:warmup_end]
    .fillna(0.5)
    .clip(0, 1)
)
probability_is_history_fallback.iloc[:warmup_end] = 1

# LEAKAGE FIX: Reuse the best Optuna LightGBM params from Section 8
forward_lgb_base_params = {
    'objective': 'binary',
    'metric': 'average_precision',
    'verbosity': -1,
    'seed': RANDOM_STATE,
    'deterministic': True,
    'force_col_wise': True,
    'bagging_freq': 1,
    **best_params_lgb
}

# LEAKAGE FIX: Pre-transform ALL raw features using the SAME preprocessor
# fitted in Section 4. This avoids per-fold OneHotEncoder drift.
X_all_transformed = transform_features(X_raw)

forward_fold_rows = []
for fold_number, (score_start, score_end) in enumerate(
    zip(fold_edges[:-1], fold_edges[1:]), start=1
):
    # Fold data: train on [0:score_start], score [score_start:score_end]
    history_end = score_start
    calibration_start = int(0.80 * history_end)
    
    fold_x_train = X_all_transformed.iloc[:calibration_start]
    fold_x_cal   = X_all_transformed.iloc[calibration_start:history_end]
    fold_x_score = X_all_transformed.iloc[score_start:score_end]
    fold_y_train = y.iloc[:calibration_start]
    fold_y_cal   = y.iloc[calibration_start:history_end]

    fold_ratio = (fold_y_train.eq(0).sum() / max(fold_y_train.eq(1).sum(), 1))
    fold_params = {
        **forward_lgb_base_params,
        'scale_pos_weight': float(0.47 * fold_ratio),
    }
    fold_train_set = lgb.Dataset(
        fold_x_train, fold_y_train, params={'feature_pre_filter': False}
    )
    fold_cal_set = lgb.Dataset(
        fold_x_cal, fold_y_cal, reference=fold_train_set,
        params={'feature_pre_filter': False},
    )
    fold_model = lgb.train(
        fold_params,
        fold_train_set,
        num_boost_round=700,
        valid_sets=[fold_cal_set],
        callbacks=[
            lgb.early_stopping(75, first_metric_only=True, verbose=False),
            lgb.log_evaluation(0),
        ],
    )
    fold_cal_raw_probability = fold_model.predict(
        fold_x_cal, num_iteration=fold_model.best_iteration
    )
    fold_score_raw_probability = fold_model.predict(
        fold_x_score, num_iteration=fold_model.best_iteration
    )

    if fold_y_cal.nunique() == 2:
        fold_calibrator = LogisticRegression(solver='lbfgs', random_state=RANDOM_STATE)
        fold_calibrator.fit(
            fold_cal_raw_probability.reshape(-1, 1), fold_y_cal
        )
        fold_probability = fold_calibrator.predict_proba(
            fold_score_raw_probability.reshape(-1, 1)
        )[:, 1]
    else:
        fold_probability = fold_score_raw_probability

    forward_probability.iloc[score_start:score_end] = fold_probability
    forward_fold_rows.append({
        'fold': fold_number,
        'training_rows': len(fold_x_train),
        'calibration_rows': len(fold_x_cal),
        'scored_rows': len(fold_x_score),
        'best_iteration': int(fold_model.best_iteration),
    })

assert forward_probability.notna().all(), 'Missing forward Model 1 probabilities.'
assert forward_probability.between(0, 1).all(), 'Invalid Model 1 probability.'

model1_handoff_scores = df[['_source_row', 'id']].copy()
model1_handoff_scores['road_closure_probability'] = forward_probability
model1_handoff_scores['road_closure_probability_is_history_fallback'] = (
    probability_is_history_fallback
)

model2_handoff_df = duration_base_df.merge(
    model1_handoff_scores,
    on=['_source_row', 'id'],
    how='left',
    validate='one_to_one',
)
assert model2_handoff_df['road_closure_probability'].notna().all()
assert len(model2_handoff_df) == len(duration_base_df)

MODEL2_HANDOFF_PATH = str(output_dir / 'model2_duration_handoff.csv')
model2_handoff_df.to_csv(MODEL2_HANDOFF_PATH, index=False)

display(pd.DataFrame(forward_fold_rows))
print('Model 2 handoff:', model2_handoff_df.shape)
print('Saved:', MODEL2_HANDOFF_PATH)